# Title

Angos and Tan

BSDSBA 2028

### Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

### Data Download

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("usdot/flight-delays", output_dir="./data")

# print("Path to dataset files:", path)

## OVERVIEW

This mini-project analyzes airline operations using the **U.S. Department of Transportation (DOT) 2015 Flight Delays and Cancellations dataset**. The project investigates how airport congestion, airline scheduling practices, and traffic density contribute to flight delays. In addition to identifying structural causes of delays, the project aims to forecast flight delay duration based on operational conditions, including airport traffic, route characteristics, and schedule realism.

Airline delays are often attributed to weather or operational disruptions, but underlying causes may include unrealistic scheduling and airport capacity constraints. By analyzing patterns in flight schedules, traffic density, and delay distributions, this project aims to identify systemic patterns that contribute to delays and congestion.

Understanding these factors has practical applications for airlines, airports, and passengers. Airlines can improve schedule planning, airports can identify congestion thresholds, and travelers can better understand which routes or airports are more reliable.


## GOALS

1. Analyze patterns in flight delays across airlines, routes, airports, and time of day.

2. Identify congestion thresholds at which airport traffic density results in significant increases in delay.

3. Forecast expected delay duration for flights using operational features such as airport traffic, route reliability, and schedule realism.

4. Evaluate whether airline schedules are realistic by comparing scheduled flight time with actual flight durations.


## Basic

### Data Loading
The raw dataset, provided by the U.S. D.O.T, and uploaded on Kaggle contains 5.8M flights across 322 airports and 14 airlines. This covers all domestic flights within the United States of America for the year of 2015.

https://www.kaggle.com/datasets/usdot/flight-delays/data

In [ ]:
# Load Data
airlines = pd.read_csv("data/airlines.csv")
airports = pd.read_csv("data/airports.csv")
flights = pd.read_csv("data/flights.csv", low_memory=False)

In [ ]:
display(flights.head())
print(flights.info())
print(flights.shape)

### DATA CLEANING

#### Fix 5-digit airport codes to standard 3-character IATA codes

Some data points use 5-digit airport codes instead of the standard IATA codes:

In [ ]:
flights[flights['ORIGIN_AIRPORT'].str.len()!=3].head()[['ORIGIN_AIRPORT','DESTINATION_AIRPORT']]

Also, comparing airports present in our `flights` and `airports` dataframe, one airport is missing. To solve this, that airport will be removed from the list of flights entirely. 

We'll utilize a helper function to fix these two issues.

In [ ]:
# import clean_flights

# clean_flights.clean('data/flights.csv')

### FEATURE ENGINEERING

The cleaned flights will be used instead of the original csv file.

In [ ]:
flights = pd.read_csv("data/flights_filtered_fixed.csv", low_memory=False)

#### Airport-level features:
Two features are computed per airport to support later analysis: average flights per hour (a proxy for traffic load) and average delay rate (probability that a departure from this airport arrives late).

- `avg_flights_per_hour`: $\frac{\text{total flights for airport}}{\text{number of hours observed}}$
- `avg_delay_rate`: P(delay)

In [ ]:
filtered_flights = flights[
    (flights['CANCELLED'] == 0) &
    (flights['DIVERTED'] == 0)
]

airports['avg_flights_per_hour'] = airports['IATA_CODE'].map(
    flights.assign(hour=filtered_flights['SCHEDULED_DEPARTURE']//100)
           .groupby(['ORIGIN_AIRPORT', 'MONTH', 'DAY', 'hour'])
           .size()
           .groupby('ORIGIN_AIRPORT')
           .mean()
)

airport_delay = (
    filtered_flights
    .assign(delayed=lambda x: x['ARRIVAL_DELAY'] > 0)
    .groupby('ORIGIN_AIRPORT')['delayed']
    .mean()
)

airports['AVG_DELAY_RATE'] = airports['IATA_CODE'].map(airport_delay)

display(airports.head())


#### Route-level features

- `schedule_padding` - $\text{average scheduled time} - \text{median flight time}$

- `ave_delay` - $\frac{\sum \text{arrival delays}}{\text{num flights for the route}}$

- `delay-rate` - P(delay)

In [ ]:
route_df = (
    filtered_flights
    .groupby(['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'], as_index=False)
    .agg(
        NUM_FLIGHTS=('ELAPSED_TIME', 'size'),
        AVE_FLIGHT_TIME=('ELAPSED_TIME', 'mean'),
        MEDIAN_FLIGHT_TIME=('ELAPSED_TIME', 'median'),
        AVE_SCHEDULED_TIME=('SCHEDULED_TIME', 'mean')
    )
)

route_df['SCHEDULE_PADDING'] = (
    route_df['AVE_SCHEDULED_TIME'] -
    route_df['MEDIAN_FLIGHT_TIME']
)

delay_stats = (
    filtered_flights
    .assign(delayed=lambda x: x['ARRIVAL_DELAY'] > 0)
    .groupby(['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])
    .agg(
        AVE_DELAY=('ARRIVAL_DELAY', 'mean'),
        DELAY_RATE=('delayed', 'mean')
    )
    .reset_index()
)

route_df = route_df.merge(
    delay_stats,
    on=['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'],
    how='left'
)

display(route_df.head())

### Delays by Airline

In [ ]:
airline_delay = (
    filtered_flights
    .assign(delayed=lambda x: x['ARRIVAL_DELAY'] > 0)
    .groupby('AIRLINE')
    .agg(
        AVE_DELAY=('ARRIVAL_DELAY', 'mean'),
        DELAY_RATE=('delayed', 'mean'),
    )
    .reset_index()
    .merge(airlines, left_on='AIRLINE', right_on='IATA_CODE')
    .sort_values('DELAY_RATE', ascending=True)  # ascending so largest is at top
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].barh(airline_delay['AIRLINE_y'], airline_delay['AVE_DELAY'], color='steelblue')
axes[0].set_title('Average Arrival Delay by Airline')
axes[0].set_xlabel('Minutes')

axes[1].barh(airline_delay['AIRLINE_y'], airline_delay['DELAY_RATE'], color='salmon')
axes[1].set_title('Delay Rate by Airline')
axes[1].set_xlabel('Proportion of flights delayed')

plt.tight_layout()
plt.show()

### Delays by Airport

In [ ]:
display(airports.sort_values(by='AVG_DELAY_RATE', ascending=False).head(10).reset_index(drop=True)[['AIRPORT','AVG_DELAY_RATE']])

In [ ]:
avg_flights_by_hour = (
    filtered_flights
    .assign(hour=filtered_flights['SCHEDULED_DEPARTURE'] // 100)
    .groupby(['ORIGIN_AIRPORT', 'MONTH', 'DAY', 'hour'])
    .size()
    .reset_index(name='flights')
    .groupby(['ORIGIN_AIRPORT', 'hour'])['flights']
    .mean()
    .reset_index(name='avg_flights')
)

delay_by_hour = (
    filtered_flights
    .assign(
        hour=filtered_flights['SCHEDULED_DEPARTURE'] // 100,
        delayed=filtered_flights['ARRIVAL_DELAY'] > 0
    )
    .groupby(['ORIGIN_AIRPORT', 'hour'])['delayed']
    .mean()
    .reset_index(name='delay_rate')
)

In [ ]:
plot_df = avg_flights_by_hour.merge(
    delay_by_hour,
    on=['ORIGIN_AIRPORT', 'hour']
)

px.scatter(
    plot_df,
    x='avg_flights',
    y='delay_rate',
    color='hour',                          # color by hour so you can see time-of-day pattern
    hover_data={'ORIGIN_AIRPORT': True},
    labels={
        'avg_flights': 'Average Flights per Hour',
        'delay_rate': 'Average Delay Rate',
        'hour': 'Hour of Day'
    },
    title='Airport Traffic Density vs Delay Rate by Hour of Day',
    trendline='ols',
    opacity=0.5,
)

In [ ]:
px.scatter(
    airports,
    x = 'avg_flights_per_hour',
    y = 'AVG_DELAY_RATE',
    labels = {'avg_flights_per_hour': 'Average Flights per Hour',
              'AVG_DELAY_RATE': 'Average Delay Rate'},
    title = 'Airport-level Traffic to delay rate relationship',
    subtitle= 'There is a slight positive trend showing that delay rates increase as airports faces higher hourly flights.',
    trendline='ols'
)

### Delay by Hour

In [ ]:
airport_hour_df = (
    filtered_flights
    .assign(
        hour=filtered_flights['SCHEDULED_DEPARTURE'] // 100,
        delayed=filtered_flights['ARRIVAL_DELAY'] > 0
    )
    .groupby(['ORIGIN_AIRPORT', 'MONTH', 'DAY', 'hour'])
    .agg(
        flights=('FLIGHT_NUMBER', 'size'),
        avg_delay=('ARRIVAL_DELAY', lambda x: x[x > 0].mean()),
        delay_rate=('delayed', 'mean')
    )
    .reset_index()
)

airport_hour_df.sort_values(by='delay_rate')

fig = px.scatter(
    airport_hour_df,
    x='hour',
    y='delay_rate',
    trendline='ols',
    opacity=0.4,
    hover_data = {'ORIGIN_AIRPORT': True}
)

fig.show()

### Route Reliability

In [ ]:
route_only = (
    route_df
    .groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'AIRLINE'])
    .agg(
        NUM_FLIGHTS=('NUM_FLIGHTS', 'sum'),
        DELAY_RATE=('DELAY_RATE', 'mean'),       # avg across airlines
        SCHEDULE_PADDING=('SCHEDULE_PADDING', 'mean'),
        AVE_DELAY=('AVE_DELAY', 'mean'),
    )
    .reset_index()
)
route_only['ROUTE'] = route_only['ORIGIN_AIRPORT'] + ' → ' + route_only['DESTINATION_AIRPORT']

route_map = route_only.merge(
    airports[['IATA_CODE', 'LATITUDE', 'LONGITUDE']],
    left_on='ORIGIN_AIRPORT', right_on='IATA_CODE'
).rename(columns={'LATITUDE': 'orig_lat', 'LONGITUDE': 'orig_lon'}).drop(columns='IATA_CODE')

route_map = route_map.merge(
    airports[['IATA_CODE', 'LATITUDE', 'LONGITUDE']],
    left_on='DESTINATION_AIRPORT', right_on='IATA_CODE'
).rename(columns={'LATITUDE': 'dest_lat', 'LONGITUDE': 'dest_lon'}).drop(columns='IATA_CODE')

ROUTE_COLORS = [
    'rgba(100,149,237,0.4)',  # blue        — <10%
    'rgba(150,200,100,0.5)',  # light green — 10–20%
    'rgba(200,230,80,0.6)',   # yellow-green— 20–30%
    'rgba(255,220,50,0.6)',   # yellow      — 30–40%
    'rgba(255,180,30,0.7)',   # gold        — 40–50%
    'rgba(255,140,0,0.7)',    # orange      — 50–60%
    'rgba(240,80,20,0.75)',   # red-orange  — 60–70%
    'rgba(220,40,40,0.8)',    # red         — 70–80%
    'rgba(160,10,10,0.87)',   # dark red    — 80–90%
    'rgba(100,0,0,0.95)',     # near black  — 90%+
]

ROUTE_BINS = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

route_map['BIN'] = pd.cut(
    route_map['DELAY_RATE'],
    bins=ROUTE_BINS,
    labels=list(range(len(ROUTE_COLORS))),
    include_lowest=True
).astype(float).fillna(0).astype(int)

In [ ]:
def plot_routes_from(origin: str = 'ALL'):
    subset = route_map if origin == 'ALL' else route_map[route_map['ORIGIN_AIRPORT'] == origin]

    if subset.empty:
        print(f"No routes found for {origin}")
        return

    fig = go.Figure()

    for i, color in enumerate(ROUTE_COLORS):
        group = subset[subset['BIN'] == i]
        lats, lons = [], []
        for _, row in group.iterrows():
            lats += [row['orig_lat'], row['dest_lat'], None]
            lons += [row['orig_lon'], row['dest_lon'], None]

        fig.add_trace(go.Scattermap(
            lat=lats, lon=lons, mode='lines',
            line=dict(width=1.5, color=color),
            hoverinfo='none', showlegend=False,
        ))

    # Colorbar
    colorbar_trace = go.Scattermap(
        lat=[None] * len(ROUTE_BINS),
        lon=[None] * len(ROUTE_BINS),
        mode='markers',
        marker=dict(
            size=0,
            color=ROUTE_BINS,
            colorscale=[
                [i / (len(ROUTE_COLORS) - 1), color]
                for i, color in enumerate(ROUTE_COLORS)
            ],
            cmin=0, cmax=1,
            colorbar=dict(
                title=dict(text='Delay Rate', font=dict(color='white')),
                tickvals=ROUTE_BINS,
                ticktext=[f'{int(b*100)}%' for b in ROUTE_BINS],
                tickfont=dict(color='white'),
                bgcolor='rgba(0,0,0,0)',
                bordercolor='rgba(255,255,255,0.3)',
                borderwidth=1,
                thickness=15,
                len=0.5,
                y=0.5,
                x=1.0,
            ),
            showscale=True,
        ),
        hoverinfo='none',
        showlegend=False,
    )

    fig.add_trace(node_trace)
    fig.add_trace(colorbar_trace)

    title = 'Route Reliability — All Routes' if origin == 'ALL' else f'Route Reliability from {origin}'

    fig.update_layout(
        map=dict(style='carto-darkmatter', center=dict(lat=39, lon=-98), zoom=3),
        margin=dict(r=80, t=50, l=0, b=0),
        title=dict(text=title, font=dict(color='white')),
        paper_bgcolor='#1a1a2e',
    )

    fig.show()

In [ ]:
plot_routes_from("ORD")
plot_routes_from("LAX")
plot_routes_from("MIA")
plot_routes_from("JFK")
plot_routes_from("SEA")
plot_routes_from()

## Intermediate
Basic analysis showed us *where* and *when* delays occur. This section investigates *why*; breaking down delays into their structural causes: airport congestion, aircraft propagation, and the interaction between them. It will then create a model to predict delays based on these two info points.

### Plotting Traffic Density to Delay

We previously computed for airport level data by the hour, including traffic volume and the delay rate.
Here we aggregate across all airports to see whether higher traffic density correlates with higher delays. 
The bar chart shows the number of flight records in each bin — taller bars indicate more reliable estimates.

In [ ]:
traffic_delay_agg = (
    airport_hour_df
    .groupby(pd.cut(airport_hour_df['flights'], 
                    bins=range(0, int(airport_hour_df['flights'].max()) + 5, 5), 
                    right=False))['avg_delay']
    .agg(['mean', 'median', 'count'])
    .reset_index()
    .rename(columns={'flights': 'TRAFFIC_BIN'})
)

traffic_delay_agg['BIN_CENTER'] = traffic_delay_agg['TRAFFIC_BIN'].apply(
    lambda x: x.mid if pd.notna(x) else np.nan
)
traffic_delay_agg.dropna(subset=['BIN_CENTER'], inplace=True)

fig, ax1 = plt.subplots(figsize=(14, 5))

ax2 = ax1.twinx()
ax2.bar(traffic_delay_agg['BIN_CENTER'], traffic_delay_agg['count'],
        width=4, color='lightblue', alpha=0.5, label='Flight Count')
ax2.set_ylabel('Number of Flights', color='steelblue')

ax1.plot(traffic_delay_agg['BIN_CENTER'], traffic_delay_agg['mean'],
         color='red', marker='o', linewidth=2, label='Mean Delay')
ax1.plot(traffic_delay_agg['BIN_CENTER'], traffic_delay_agg['median'],
         color='darkorange', marker='s', linestyle='--', linewidth=2, label='Median Delay')
ax1.set_xlabel('Flights Per Hour (Traffic Density)')
ax1.set_ylabel('Arrival Delay (minutes)', color='red')
ax1.set_title('Airport Traffic Density vs. Mean Arrival Delay')
ax1.legend(loc='upper left')

plt.tight_layout()
plt.show()

### Congestion Thresholds
The aggregate view masks airport-to-airport differences. Each airport has its own capacity, 
a point at which adding more flights starts baking a baseline delay into every departure, 
regardless of whether the individual aircraft is late.

We identify this threshold per airport as the traffic level where the *floor* of delay rate 
(10th percentile) rises above near-zero — indicating congestion rather than random plane-level causes.

In [ ]:
def find_threshold_rising_floor(group):
    group = group[group['flights'] >= 5].sort_values('flights').dropna(subset=['flights', 'delay_rate'])
    if len(group) < 15:
        return None
    
    group['bin'] = pd.qcut(group['flights'], q=10, duplicates='drop')
    binned = group.groupby('bin', observed=True).agg(
        floor=('delay_rate', lambda x: x.quantile(0.1)),
        flights=('flights', 'mean'),
    )
    
    rising = binned[binned['floor'] > 0.05]
    if rising.empty:
        return None
    return rising.iloc[0]['flights']

thresholds = (
    airport_hour_df
    .groupby('ORIGIN_AIRPORT')
    .apply(find_threshold_rising_floor, include_groups=False)
    .dropna()
)

print(thresholds.describe())
print(f"\nAirports with thresholds: {len(thresholds)} / {airport_hour_df['ORIGIN_AIRPORT'].nunique()}")

# Visualize threshold distribution
fig, ax = plt.subplots(figsize=(10, 4))
thresholds.hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Congestion threshold (flights/hour)')
ax.set_ylabel('Number of airports')
ax.set_title('Distribution of Congestion Thresholds Across Airports')
plt.tight_layout()
plt.show()

In [ ]:
airport = 'MSP'
group = airport_hour_df[airport_hour_df['ORIGIN_AIRPORT'] == airport].dropna(subset=['flights', 'delay_rate'])

x = np.linspace(group['flights'].min(), group['flights'].max(), 200)
plt.scatter(group['flights'], group['delay_rate'], alpha=0.4, label='data')
plt.axvline(thresholds[airport], color='red', linestyle='--', label=f'threshold: {thresholds[airport]:.0f} flights')
plt.xlabel('flights per hour')
plt.ylabel('delay rate')
plt.title(f'{airport} capacity threshold')
plt.legend()
plt.show()

### Delay Propagation Through the Aircraft

Airport congestion is one source of delay. A second, independent source is the aircraft itself — 
a late-arriving plane causes its next departure to be late too, cascading through the day.

We track each aircraft by tail number, linking consecutive legs to measure how much delay 
is inherited vs absorbed at each turnaround.

In [ ]:
# Sort each aircraft's flights in order
chains = filtered_flights[
    filtered_flights['TAIL_NUMBER'].notna() &
    filtered_flights['ARRIVAL_DELAY'].notna() &
    filtered_flights['DEPARTURE_DELAY'].notna()
].sort_values(['TAIL_NUMBER', 'YEAR', 'MONTH', 'DAY', 'DEPARTURE_TIME']).copy()

# Shift to get previous leg info
chains['PREV_ARRIVAL_DELAY']  = chains.groupby('TAIL_NUMBER')['ARRIVAL_DELAY'].shift(1)
chains['PREV_DESTINATION']    = chains.groupby('TAIL_NUMBER')['DESTINATION_AIRPORT'].shift(1)
chains['LEG']                 = chains.groupby('TAIL_NUMBER').cumcount()

# Only keep legs where the previous destination matches current origin
# (filters out overnight resets / data gaps)
chains = chains[
    chains['PREV_DESTINATION'] == chains['ORIGIN_AIRPORT']
].copy()

In [ ]:
# How much delay was absorbed vs passed on at turnaround
chains['DELAY_DELTA'] = chains['DEPARTURE_DELAY'] - chains['PREV_ARRIVAL_DELAY']
# Negative = airport/crew absorbed delay
# Positive = delay grew at turnaround

# Was this flight's departure delay mostly explained by the previous arrival?
chains['PROPAGATED'] = (
    (chains['PREV_ARRIVAL_DELAY'] > 0) &
    (chains['DEPARTURE_DELAY'] > 0)
)

# How much of the departure delay came from the inbound aircraft?
chains['INHERITED_RATIO'] = (
    chains['DEPARTURE_DELAY'] / chains['PREV_ARRIVAL_DELAY']
).clip(0, 1)  # cap at 1 (can't inherit more than 100%)
chains['INHERITED_RATIO'] = chains['INHERITED_RATIO'].where(
    chains['PREV_ARRIVAL_DELAY'] > 0, 0
)

In [ ]:
print("=== Propagation Rate ===")
print(chains['PROPAGATED'].value_counts(normalize=True))

print("\n=== Avg delay delta at turnaround ===")
print(chains['DELAY_DELTA'].describe())

print("\n=== Worst propagating aircraft ===")
print(
    chains[chains['PROPAGATED']]
    .groupby('TAIL_NUMBER')
    .agg(
        LEGS=('FLIGHT_NUMBER', 'count'),
        AVG_INHERITED=('PREV_ARRIVAL_DELAY', 'mean'),
        AVG_DEPARTED=('DEPARTURE_DELAY', 'mean'),
        AVG_DELTA=('DELAY_DELTA', 'mean'),
    )
    .sort_values('LEGS', ascending=False)
    .head(20)
)

print("\n=== Routes most affected by propagation ===")
print(
    chains[chains['PROPAGATED']]
    .groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])
    .agg(
        PROPAGATED_FLIGHTS=('PROPAGATED', 'sum'),
        AVG_DELAY=('DEPARTURE_DELAY', 'mean'),
    )
    .sort_values('PROPAGATED_FLIGHTS', ascending=False)
    .head(20)
)

### Delay Propagation Through the Airport

While tail number propagation follows individual aircraft, airport-level propagation occurs when 
a busy hub accumulates delayed inbound aircraft faster than it can turn them around.

The turnaround window — the scheduled time between a plane's arrival and its next departure — 
is the key buffer. We match each inbound leg to the next outbound leg of the same aircraft at 
the same airport using scheduled times to identify this turnaround window.

In [ ]:
# Get the inbound leg — when was it scheduled to arrive?
inbound_sched = (
    filtered_flights[['TAIL_NUMBER', 'DESTINATION_AIRPORT', 'MONTH', 'DAY',
                   'SCHEDULED_ARRIVAL', 'ARRIVAL_DELAY']]
    .rename(columns={'DESTINATION_AIRPORT': 'AIRPORT',
                     'SCHEDULED_ARRIVAL': 'SCHED_ARR'})
)

# Get the outbound leg — when was it scheduled to depart?
outbound_sched = (
    filtered_flights[['TAIL_NUMBER', 'ORIGIN_AIRPORT', 'MONTH', 'DAY',
                   'SCHEDULED_DEPARTURE', 'DEPARTURE_DELAY']]
    .rename(columns={'ORIGIN_AIRPORT': 'AIRPORT',
                     'SCHEDULED_DEPARTURE': 'SCHED_DEP'})
)

# Join on tail number + airport + day — same aircraft, same airport
turnaround = inbound_sched.merge(
    outbound_sched,
    on=['TAIL_NUMBER', 'AIRPORT', 'MONTH', 'DAY'],
    how='inner'
)

# Keep only rows where the departure was scheduled AFTER the arrival
# and within a reasonable turnaround window (e.g. 30–180 min)
turnaround['SCHED_ARR'] = pd.to_numeric(turnaround['SCHED_ARR'], errors='coerce')
turnaround['SCHED_DEP'] = pd.to_numeric(turnaround['SCHED_DEP'], errors='coerce')

turnaround['TURNAROUND_MIN'] = (
    (turnaround['SCHED_DEP'] // 100 * 60 + turnaround['SCHED_DEP'] % 100) -
    (turnaround['SCHED_ARR'] // 100 * 60 + turnaround['SCHED_ARR'] % 100)
)

turnaround = turnaround[
    turnaround['TURNAROUND_MIN'].between(30, 180)
]

# Now transmission is direct — did delayed arrival cause delayed departure?
turnaround['PROPAGATED'] = (
    (turnaround['ARRIVAL_DELAY'] > 15) &
    (turnaround['DEPARTURE_DELAY'] > 0)
)

print(turnaround['TURNAROUND_MIN'].describe())
print(f"\nPropagation rate: {turnaround['PROPAGATED'].mean():.1%}")

print("\n=== Worst airports for propagation ===")
print(
    turnaround[turnaround['PROPAGATED']]
    .groupby('AIRPORT')
    .agg(
        PROPAGATED_FLIGHTS=('PROPAGATED', 'sum'),
        AVG_TURNAROUND=('TURNAROUND_MIN', 'mean'),
        AVG_ARRIVAL_DELAY=('ARRIVAL_DELAY', 'mean'),
        AVG_DEPARTURE_DELAY=('DEPARTURE_DELAY', 'mean'),
    )
    .sort_values('PROPAGATED_FLIGHTS', ascending=False)
    .head(20)
)

### Decomposing Delay: Plane-Caused vs Airport-Caused

With both propagation mechanisms identified, we decompose each departure delay into two components:

- **Plane-caused**: the portion explained by the inbound aircraft arriving late
- **Residual**: what remains after accounting for the aircraft — attributed to airport-level factors

If the residual grows as traffic increases while plane-caused stays flat, it confirms that 
congestion is progressively taking over from aircraft cascades as the dominant delay mechanism.

In [ ]:
# Step 1: From the turnaround analysis, get plane-level delay per flight
plane_delay = turnaround[['TAIL_NUMBER', 'AIRPORT', 'MONTH', 'DAY', 
                           'SCHED_DEP', 'DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'PROPAGATED']].copy()

# How much of the departure delay is explained by the inbound aircraft?
plane_delay['PLANE_CAUSED'] = plane_delay['ARRIVAL_DELAY'].clip(lower=0)
plane_delay['PLANE_CAUSED'] = plane_delay[['DEPARTURE_DELAY', 'PLANE_CAUSED']].min(axis=1)

# Residual = what's left after accounting for the plane
plane_delay['RESIDUAL_DELAY'] = (plane_delay['DEPARTURE_DELAY'] - plane_delay['PLANE_CAUSED']).clip(lower=0)

print(plane_delay[['DEPARTURE_DELAY', 'PLANE_CAUSED', 'RESIDUAL_DELAY']].describe())

In [ ]:
plane_delay['hour'] = pd.to_numeric(plane_delay['SCHED_DEP'], errors='coerce') // 100

# Aggregate to airport-hour level
plane_hour = (
    plane_delay
    .groupby(['AIRPORT', 'MONTH', 'DAY', 'hour'])
    .agg(
        avg_plane_delay=('PLANE_CAUSED', 'mean'),
        avg_residual=('RESIDUAL_DELAY', 'mean'),
        prop_plane_caused=('PLANE_CAUSED', lambda x: (x > 0).mean()),
    )
    .reset_index()
)

# Merge with airport_hour_df
merged = airport_hour_df.merge(
    plane_hour,
    left_on=['ORIGIN_AIRPORT', 'MONTH', 'DAY', 'hour'],
    right_on=['AIRPORT', 'MONTH', 'DAY', 'hour'],
    how='left'
).merge(
    thresholds.rename('threshold'),
    left_on='ORIGIN_AIRPORT', right_index=True,
    how='left'
)

merged['CONGESTED'] = merged['flights'] > merged['threshold']

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Plane vs residual delay split
merged.groupby('CONGESTED')[['avg_plane_delay', 'avg_residual']].mean().plot(
    kind='bar', ax=axes[0], title='Plane vs Residual Delay\n(below vs above threshold)'
)
axes[0].set_xticklabels(['Below threshold', 'Above threshold'], rotation=0)
axes[0].set_ylabel('avg minutes')

# Plot 2: As congestion grows, does residual grow faster than plane delay?
merged['flight_bin'] = pd.qcut(merged['flights'], q=20, duplicates='drop')
binned = merged.groupby('flight_bin', observed=True)[['avg_plane_delay', 'avg_residual']].mean()
binned.plot(ax=axes[1], title='Delay components vs traffic volume')
axes[1].set_xlabel('flights per hour (binned)')
axes[1].set_ylabel('avg minutes')

# Plot 3: Proportion of delay that is plane-caused vs congestion
merged['pct_plane'] = merged['avg_plane_delay'] / (merged['avg_plane_delay'] + merged['avg_residual']).replace(0, None)
merged.groupby('flight_bin', observed=True)['pct_plane'].mean().plot(
    ax=axes[2], title='% of delay from plane vs airport\nas traffic increases'
)
axes[2].axhline(0.5, color='red', linestyle='--', label='50/50 split')
axes[2].set_ylabel('proportion plane-caused')
axes[2].set_xlabel('flights per hour (binned)')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Classify airports by max traffic
airport_max_flights = merged.groupby('ORIGIN_AIRPORT')['flights'].max()
large  = airport_max_flights[airport_max_flights >= 17].index
medium = airport_max_flights[airport_max_flights.between(8, 17)].index
small  = airport_max_flights[airport_max_flights < 8].index

merged['AIRPORT_SIZE'] = pd.cut(
    merged['ORIGIN_AIRPORT'].map(airport_max_flights),
    bins=[0, 8, 17, float('inf')],
    labels=['small', 'medium', 'large']
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (size, group) in enumerate(merged.groupby('AIRPORT_SIZE', observed=True)):
    group['flight_bin'] = pd.qcut(group['flights'], q=10, duplicates='drop')
    binned = group.groupby('flight_bin', observed=True)[['avg_plane_delay', 'avg_residual']].mean()
    binned.plot(ax=axes[i], title=f'{size} airports')
    axes[i].set_xlabel('flights per hour')
    axes[i].set_ylabel('avg minutes')

plt.tight_layout()
plt.show()

### Regression Model

Using the features derived above — traffic density, congestion status, schedule padding, 
route reliability, and departure delay — we build a model to forecast arrival delay duration.

Two models are compared: a linear regression baseline and a gradient boosting model. 
Feature importance from the gradient booster reveals which structural factors matter most 
beyond the obvious signal of departure delay.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

# --- Assemble features ---
model_df = filtered_flights.copy()
model_df['hour'] = model_df['SCHEDULED_DEPARTURE'] // 100

model_df = model_df.merge(
    airport_hour_df[['ORIGIN_AIRPORT', 'MONTH', 'DAY', 'hour', 'flights', 'delay_rate']]
    .rename(columns={'flights': 'FLIGHTS_PER_HOUR', 'delay_rate': 'AIRPORT_DELAY_RATE'}),
    on=['ORIGIN_AIRPORT', 'MONTH', 'DAY', 'hour'],
    how='left'
)

model_df = model_df.merge(
    route_df[['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT',
              'SCHEDULE_PADDING', 'AVE_DELAY', 'DELAY_RATE']],
    on=['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'],
    how='left'
)

model_df = model_df.merge(
    thresholds.rename('THRESHOLD'),
    left_on='ORIGIN_AIRPORT', right_index=True,
    how='left'
)

le = LabelEncoder()
model_df['AIRLINE_ENC'] = le.fit_transform(model_df['AIRLINE'].astype(str))

features = [
    'FLIGHTS_PER_HOUR',
    'hour',
    'MONTH',
    'SCHEDULE_PADDING',
    'AVE_DELAY',
    'DELAY_RATE',
    'DISTANCE',
    'AIRLINE_ENC',
    'DEPARTURE_DELAY',
]
target = 'ARRIVAL_DELAY'

model_df = model_df.dropna(subset=features + [target])
model_df = model_df[model_df[target].between(-60, 500)]

X = model_df[features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline
lr = LinearRegression().fit(X_train, y_train)
lr_pred = lr.predict(X_test)
print("=== Linear Regression ===")
print(f"RMSE: {mean_squared_error(y_test, lr_pred):.2f} min")
print(f"R²:   {r2_score(y_test, lr_pred):.3f}")

# Gradient boosting
gb = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
print("=== Gradient Boosting ===")
print(f"RMSE: {mean_squared_error(y_test, gb_pred):.2f} min")
print(f"R²:   {r2_score(y_test, gb_pred):.3f}")

# Feature importance
importance = pd.Series(gb.feature_importances_, index=features).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 5))
importance.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importance (Gradient Boosting)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## Advanced

## Visualizing Flights
The following code renders a graphic of flights by hour

In [ ]:
# Graphing every airport as points on the map
node_trace = go.Scattermap(
    lat=airports['LATITUDE'],
    lon=airports['LONGITUDE'],
    mode='markers',
    marker=dict(size=5, color='white', opacity=0.8),
    text=airports['IATA_CODE'] + ' — ' + airports['CITY'],
    hoverinfo='text',
    showlegend=False,
)

# Delay color/width bins 
DELAY_BINS   = [-np.inf, 0, 10, 20, 30, 40, 50, 60, 70, 80, 90, np.inf]
DELAY_COLORS = [
    'rgba(100,149,237,0.5)',  # blue       — on time
    'rgba(200,230,80,0.6)',   # yellow-green — 1–10
    'rgba(255,220,50,0.6)',   # yellow     — 10–20
    'rgba(255,180,30,0.7)',   # gold       — 20–30
    'rgba(255,140,0,0.7)',    # orange     — 30–40
    'rgba(240,80,20,0.75)',   # red-orange — 40–50
    'rgba(220,40,40,0.8)',    # red        — 50–60
    'rgba(190,20,20,0.85)',   # dark red   — 60–70
    'rgba(160,10,10,0.87)',   # darker red — 70–80
    'rgba(139,0,0,0.9)',      # deep red   — 80–90
    'rgba(100,0,0,0.95)',     # near black — 90+
]
DELAY_WIDTHS = [0.5, 1, 2, 3, 4.5]

In [ ]:
def build_edge_traces(hour_flights):
    # initialize all 6 traces as empty
    on_time_lats, on_time_lons = [], []
    bin_lats = {(w, c): [] for w in DELAY_WIDTHS for c in range(len(DELAY_COLORS))}
    bin_lons = {(w, c): [] for w in DELAY_WIDTHS for c in range(len(DELAY_COLORS))}

    if not hour_flights.empty:
        routes = (
            hour_flights.groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])
            .agg(COUNT=('FLIGHT_NUMBER', 'count'), AVG_DELAY=('ARRIVAL_DELAY', 'mean'))
            .reset_index()
        )
        

        routes = routes.merge(
            airports[['IATA_CODE', 'LATITUDE', 'LONGITUDE']],
            left_on='ORIGIN_AIRPORT', right_on='IATA_CODE'
        ).rename(columns={'LATITUDE': 'orig_lat', 'LONGITUDE': 'orig_lon'}).drop(columns='IATA_CODE')

        routes = routes.merge(
            airports[['IATA_CODE', 'LATITUDE', 'LONGITUDE']],
            left_on='DESTINATION_AIRPORT', right_on='IATA_CODE'
        ).rename(columns={'LATITUDE': 'dest_lat', 'LONGITUDE': 'dest_lon'}).drop(columns='IATA_CODE')

        routes['DELAYED_COUNT'] = (
            hour_flights[hour_flights['ARRIVAL_DELAY'] > 5]
            .groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])
            .size()
            .reindex(pd.MultiIndex.from_frame(routes[['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']]))
            .fillna(0)
            .values
        )

        on_time = routes[routes['DELAYED_COUNT'] == 0]
        delayed = routes[routes['DELAYED_COUNT'] > 0].copy()

        for _, row in on_time.iterrows():
            on_time_lats += [row['orig_lat'], row['dest_lat'], None]
            on_time_lons += [row['orig_lon'], row['dest_lon'], None]

        if not delayed.empty:
            # Width from delayed count
            delayed['width'] = pd.cut(
                delayed['DELAYED_COUNT'], bins=5,
                labels=DELAY_WIDTHS,
                duplicates='drop'
            ).astype(float).fillna(1)

            # Color from avg delay duration
            delayed['color_idx'] = pd.cut(
                delayed['AVG_DELAY'],
                bins=DELAY_BINS,
                labels=list(range(len(DELAY_COLORS)))
            ).astype(float).fillna(0).astype(int)

            for _, row in delayed.iterrows():
                w = row['width']
                c = row['color_idx']
                bin_lats[(w, c)] += [row['orig_lat'], row['dest_lat'], None]
                bin_lons[(w, c)] += [row['orig_lon'], row['dest_lon'], None]


    traces = [
        go.Scattermap(lat=on_time_lats, lon=on_time_lons, mode='lines',
                      line=dict(width=0.5, color='rgba(100,149,237,0.2)'),
                      hoverinfo='none', showlegend=False),
    ]
    
    for w in DELAY_WIDTHS:
        for c, color in enumerate(DELAY_COLORS):
            traces.append(go.Scattermap(
                lat=bin_lats[(w, c)], lon=bin_lons[(w, c)], mode='lines',
                line=dict(width=w, color=color),
                hoverinfo='none', showlegend=False,
            ))

    return traces  

In [ ]:
def generate_frames(flights):
    frames = []
    slider_steps = []

    min_day = int(flights['DAY'].min())
    max_day = int(flights['DAY'].max())

    for day in range(min_day, max_day + 1):
        for hour in range(24):
            label = f"Jan {day:02d}  {hour:02d}:00"
            slice_ = flights[(flights['DAY'] == day) & (flights['DEPARTURE_HOUR'] == hour)]
            edge_traces = build_edge_traces(slice_)

            frames.append(go.Frame(
                data=edge_traces + [node_trace],
                name=label
            ))
            slider_steps.append(dict(
                method='animate',
                args=[[label], dict(mode='immediate', frame=dict(duration=0, redraw=True))],
                label=f"D{day} {hour:02d}h"
            ))

    return frames, slider_steps

In [ ]:
def build_figure(flights):
    flights["DEPARTURE_HOUR"]=flights['DEPARTURE_TIME'] // 100
    frames, slider_steps = generate_frames(flights)

    min_day  = int(flights['DAY'].min())
    min_hour = int(flights['DEPARTURE_HOUR'].min())

    init_traces = build_edge_traces(
        flights[(flights['DAY'] == min_day) & (flights['DEPARTURE_HOUR'] == min_hour)]
    )

    fig = go.Figure(
        data=init_traces + [node_trace],
        frames=frames
    )

    _ = fig.update_layout(
        map=dict(style='carto-darkmatter', center=dict(lat=39, lon=-98), zoom=3),
        margin=dict(r=0, t=50, l=0, b=0),
        title=dict(text='US Flight Delays — Jan 1–7 2015 (Hourly)', font=dict(color='white')),
        paper_bgcolor='#1a1a2e',
        updatemenus=[dict(
            type='buttons', showactive=False, y=1.08, x=0.0,
            buttons=[
                dict(label='▶ Play', method='animate',
                     args=[None, dict(frame=dict(duration=400, redraw=True), fromcurrent=True)]),
                dict(label='⏸ Pause', method='animate',
                     args=[[None], dict(mode='immediate')]),
            ]
        )],
        sliders=[dict(
            steps=slider_steps,
            currentvalue=dict(prefix='Time: ', font=dict(size=13, color='white')),
            font=dict(color='white'),
            len=0.95, x=0.02,
        )]
    )

    return fig

### Overview of Flights

In [ ]:
week = flights[(flights["MONTH"] == 1) & (flights["DAY"].between(1, 7))]
build_figure(week)

### Delay Chains Visualized

In [ ]:
def chain_consistency(group):
    group = group.sort_values('DEPARTURE_TIME')
    delays = group['DEPARTURE_DELAY'].values
    if len(delays) < 3:
        return None
    
    # Count how many consecutive leg pairs show increasing delay
    increasing = sum(delays[i+1] > delays[i] for i in range(len(delays)-1))
    consistency = increasing / (len(delays) - 1)  # proportion of legs that grew
    
    return pd.Series({
        'LEGS': len(delays),
        'CONSISTENCY': consistency,          # 1.0 = delay grew every single leg
        'START_DELAY': delays[0],
        'END_DELAY': delays[-1],
        'TOTAL_GROWTH': delays[-1] - delays[0],
        'AIRPORTS': list(group['ORIGIN_AIRPORT']) + [group['DESTINATION_AIRPORT'].iloc[-1]],
    })

chain_summary = (
    chains[chains['PROPAGATED']]
    .groupby(['TAIL_NUMBER', 'MONTH', 'DAY'])
    .apply(chain_consistency, include_groups=False)
    .dropna()
    .reset_index()
)

compelling = chain_summary[
    (chain_summary['LEGS'] >= 3) &
    (chain_summary['CONSISTENCY'] >= 0.8) &   # delay grew on 80%+ of legs
    (chain_summary['START_DELAY'] > 10) &      # started with a real delay
    (chain_summary['TOTAL_GROWTH'] > 15)       # overall delay meaningfully grew
].sort_values(['CONSISTENCY', 'LEGS'], ascending=[False, False])

print(compelling.head(10))

In [ ]:
case = chains[
    (chains['TAIL_NUMBER'] == 'N353SW') &
    (chains['MONTH'] == 8) &
    (chains['DAY'] == 7)
].sort_values('DEPARTURE_TIME')

print(case[['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'DEPARTURE_TIME', 
            'PREV_ARRIVAL_DELAY', 'DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'DELAY_DELTA']])

In [ ]:
build_figure(case)